# 02 Cohort Construction

This notebook contains **Steps 2A–2C** of the workflow:

- **2A**: build the heart-failure cohort with a 30-day readmission label
- **2B**: identify admissions to exclude because of in-hospital or 30-day mortality
- **2C**: recreate the cohort after applying those exclusions


In [1]:
# Ensure this notebook can run independently by connecting to DuckDB
import duckdb
import os
import pandas as pd # often used alongside

from utils import get_db_connection
con = get_db_connection()


Connected to DuckDB at: /Users/sameer/Documents/DataScience_Capstone_Project/Capstone_Healthcare_Decision_Intelligence_Agent/dataset/hf_project.duckdb


# Step 2A: Create Heart Failure Cohort with 30-Day Readmission Label

In [2]:
con.execute("""
    CREATE OR REPLACE TABLE hf_labeled AS

    WITH hf_admissions AS (
        -- Get all HF admissions with valid admit/discharge times
        SELECT
            a.subject_id,
            a.hadm_id,
            a.admittime,
            a.dischtime
        FROM heart_diagnoses hd
        JOIN admissions a ON hd.hadm_id = a.hadm_id
        WHERE a.dischtime >= a.admittime  -- remove 175 invalid rows
    )

    SELECT
        h.subject_id,
        h.hadm_id,
        h.admittime,
        h.dischtime,
        CASE
            WHEN MIN(next_a.admittime) IS NOT NULL THEN 1
            ELSE 0
        END AS readmitted_30d

    FROM hf_admissions h
    LEFT JOIN admissions next_a
        ON  h.subject_id    = next_a.subject_id
        AND next_a.admittime > h.dischtime
        AND next_a.admittime <= h.dischtime + INTERVAL 30 DAYS
        AND next_a.hadm_id  != h.hadm_id

    GROUP BY h.subject_id, h.hadm_id, h.admittime, h.dischtime
""")

# Verify
result = con.execute("""
    SELECT
        COUNT(*)                          AS total_admissions,
        COUNT(DISTINCT subject_id)        AS unique_patients,
        SUM(readmitted_30d)               AS readmitted,
        ROUND(AVG(readmitted_30d) * 100, 2) AS readmission_rate_pct
    FROM hf_labeled
""").fetchdf()

print("hf_labeled created!")
print(result.to_string(index=False))

hf_labeled created!
 total_admissions  unique_patients  readmitted  readmission_rate_pct
             4761             4289      1034.0                 21.72


Built the `hf_labeled` table by joining `heart_diagnoses` with `admissions`, filtering out 175 invalid rows where discharge time precedes admit time. For each HF admission, computes the binary label `readmitted_30d` (1 = patient was readmitted within 30 days of discharge). Result: 4,761 admissions from 4,289 unique patients with a 21.72% readmission rate.

# Step 2B: Checked deaths in HF cohort

In [3]:
died_check = con.execute("""
    SELECT
        COUNT(*) AS total_hf_admissions,
        SUM(CASE
            WHEN p.dod IS NOT NULL
             AND CAST(p.dod AS DATE) <= CAST(h.dischtime AS DATE)
            THEN 1 ELSE 0
        END) AS died_during_admission,
        SUM(CASE
            WHEN p.dod IS NOT NULL
             AND CAST(p.dod AS DATE) <= CAST(h.dischtime AS DATE) + 30
            THEN 1 ELSE 0
        END) AS died_within_30d
    FROM hf_labeled h
    JOIN patients p ON h.subject_id = p.subject_id
""").fetchdf()

print(died_check.to_string(index=False))

 total_hf_admissions  died_during_admission  died_within_30d
                4761                  105.0            253.0


Identified patients who died during admission (105) or within 30 days of discharge (253). These cases need to be excluded because patients who died cannot be "readmitted", including them would bias the readmission prediction model.

#Step 2C: Recreate Cohort Excluding Deaths Within 30 Days

In [4]:
con.execute("""
    CREATE OR REPLACE TABLE hf_labeled AS

    WITH hf_admissions AS (
        SELECT a.subject_id, a.hadm_id, a.admittime, a.dischtime
        FROM heart_diagnoses hd
        JOIN admissions a ON hd.hadm_id = a.hadm_id
        JOIN patients p ON a.subject_id = p.subject_id
        WHERE a.dischtime >= a.admittime
          AND (p.dod IS NULL
               OR CAST(p.dod AS DATE) > CAST(a.dischtime AS DATE) + 30)
    )

    SELECT
        h.subject_id,
        h.hadm_id,
        h.admittime,
        h.dischtime,
        CASE
            WHEN MIN(next_a.admittime) IS NOT NULL THEN 1
            ELSE 0
        END AS readmitted_30d

    FROM hf_admissions h
    LEFT JOIN admissions next_a
        ON  h.subject_id    = next_a.subject_id
        AND next_a.admittime > h.dischtime
        AND next_a.admittime <= h.dischtime + INTERVAL 30 DAYS
        AND next_a.hadm_id  != h.hadm_id

    GROUP BY h.subject_id, h.hadm_id, h.admittime, h.dischtime
""")

# Verify
result = con.execute("""
    SELECT
        COUNT(*)                             AS total_admissions,
        COUNT(DISTINCT subject_id)           AS unique_patients,
        SUM(readmitted_30d)                  AS readmitted,
        ROUND(AVG(readmitted_30d) * 100, 2)  AS readmission_rate_pct
    FROM hf_labeled
""").fetchdf()

print(" hf_labeled recreated (deaths excluded)!")
print(result.to_string(index=False))

 hf_labeled recreated (deaths excluded)!
 total_admissions  unique_patients  readmitted  readmission_rate_pct
             4508             4074       971.0                 21.54


Recreated `hf_labeled` by excluding patients who died during admission or within 30 days of discharge. This is a critical clinical step dead patients cannot be readmitted, and leaving them in the dataset would introduce data leakage. Final cohort: 4,508 admissions, 4,074 unique patients, 971 readmitted (21.54% readmission rate).

In [5]:
con.close()